In [ ]:
import pandas as pd

df = pd.read_csv(".../data/raw/accepted_2007_to_2018Q4.csv", low_memory=False)

df.shape
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.info()
df = df.sample(n=100000, random_state=42)
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 2.5+ GB


(100000, 151)

In [6]:
drop_cols = [
    "id", "member_id",
    "url", "title", "zip_code",
    "emp_title", "desc"
]

df = df.drop(columns=drop_cols, errors="ignore")

In [7]:
leakage_cols = [
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_amnt"
]

df = df.drop(columns=leakage_cols, errors="ignore")

In [8]:
def classify_default(status):
    if status in ["Charged Off", "Default", "Late (31-120 days)", "Late (16-30 days)"]:
        return 1
    else:
        return 0

df["default"] = df["loan_status"].apply(classify_default)

In [9]:
df["default"].value_counts(normalize=True)
missing = df.isnull().mean().sort_values(ascending=False)
missing.head(15)

orig_projected_additional_accrued_interest    0.99617
hardship_dpd                                  0.99510
deferral_term                                 0.99510
hardship_amount                               0.99510
hardship_status                               0.99510
hardship_reason                               0.99510
hardship_last_payment_amount                  0.99510
hardship_length                               0.99510
payment_plan_start_date                       0.99510
hardship_type                                 0.99510
hardship_start_date                           0.99510
hardship_end_date                             0.99510
hardship_payoff_balance_amount                0.99510
hardship_loan_status                          0.99510
settlement_percentage                         0.98415
dtype: float64

In [10]:
post_event_cols = [col for col in df.columns if "hardship" in col or "settlement" in col]

df = df.drop(columns=post_event_cols, errors="ignore")

In [11]:
missing = df.isnull().mean()

high_missing_cols = missing[missing > 0.7].index

df = df.drop(columns=high_missing_cols)

In [12]:
keep_cols = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "application_type",
    "mort_acc",
    "pub_rec_bankruptcies",
    "default"
]

df = df[[col for col in keep_cols if col in df.columns]]

In [13]:
df.shape
df.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,application_type,mort_acc,pub_rec_bankruptcies,default
392949,32000.0,60 months,10.49,687.65,B,B3,10+ years,MORTGAGE,120000.0,Verified,...,0.0,20.0,0.0,39687.0,57.8,42.0,Individual,2.0,0.0,0
1273506,9600.0,36 months,12.99,323.42,C,C1,NaN,RENT,21900.0,Verified,...,1.0,13.0,1.0,4509.0,38.9,20.0,Individual,0.0,1.0,0
324024,4000.0,36 months,6.68,122.93,A,A3,4 years,MORTGAGE,83000.0,Not Verified,...,0.0,16.0,0.0,1564.0,17.2,25.0,Individual,2.0,0.0,0
2066630,6025.0,36 months,10.91,197.00,B,B4,10+ years,RENT,52000.0,Not Verified,...,1.0,11.0,0.0,2706.0,12.8,25.0,Individual,0.0,0.0,0
477199,25000.0,60 months,26.30,752.96,E,E5,10+ years,OWN,65000.0,Verified,...,1.0,19.0,0.0,49461.0,24.7,33.0,Individual,0.0,0.0,0


In [15]:
df["default"].mean()
df.groupby("grade")["default"].mean().sort_values()
df.groupby("purpose")["default"].mean().sort_values()
df.groupby(pd.qcut(df["annual_inc"], 5), observed=True)["default"].mean()
df.groupby(pd.qcut(df["int_rate"], 5), observed=True)["default"].mean()

int_rate
(5.308999999999999, 8.49]    0.040753
(8.49, 11.49]                0.085291
(11.49, 13.67]               0.125230
(13.67, 16.99]               0.167888
(16.99, 30.99]               0.248359
Name: default, dtype: float64

In [17]:
df.isnull().mean().sort_values(ascending=False).head(10)

emp_length              0.06458
mort_acc                0.02122
dti                     0.00084
revol_util              0.00066
pub_rec_bankruptcies    0.00052
loan_amnt               0.00004
term                    0.00004
installment             0.00004
int_rate                0.00004
annual_inc              0.00004
dtype: float64

In [18]:
df.to_csv("../data/processed/cleaned_data.csv", index=False)